# Paper figures with dinostics

Every figure and table of the paper, recomputed with the dinostics package (`Results/a_posteriori/dinostics`). Each section follows one script of this folder: what it loads and computes is shown inline (its `compute` and the helpers it calls), then the plotting function of the script draws the result (`plot(computed)`, `table(computed)` for a table). The figures are saved only by running the scripts themselves (`python <script>.py`, into `figures/` or PAPER_FIGURES). The shared helpers are in `common.py`; the call graph and the pieces dinostics has no routine for are in `CALL_GRAPH.md`. This notebook is generated by `build_notebook.py` from the scripts, edit them rather than the cells.

## fig_huber

Inline figure of Section 3: the one-sided Huber penalty h_tau of the increasing density constraint (`NegativeGradDensityHuber` in `Diffusion_Model/pipelines/losses.py`), as a function of the vertical density step delta, in units of tau. Standalone: needs no data and no dinostics. Saved at the final width (about 1.9 in) with 7-8 pt fonts in the `figures` folder of the working directory (PAPER_FIGURES), this folder when run from it.

In [ ]:
from fig_huber import *


def h(delta, tau=1.):
    """One-sided Huber: 0 for delta >= 0, delta^2 / (2 tau) on (-tau, 0), -delta - tau / 2 below -tau."""
    return np.where(delta >= 0, 0., np.where(delta < -tau, -delta - tau / 2, delta**2 / (2 * tau)))

In [ ]:
delta = np.linspace(-3, 1.5, 500)
computed = {'delta': delta, 'h': h(delta)}

In [ ]:
plot(computed);

## Loading

`from common import *` loads, once:

- `GRID`: the dinostics `NemoExperiment` of the 1° reference run (`domain_cfg_out.nc`, `mesh_mask.nc`, `namelist_cfg` of `1deg_restart`), the T grid of the generated states, its xgcm grid (`GRID.grid`) and its masks; sigma_0 is the TEOS-10 polynomial of `metrics.get_density_at_surface`, the EOS of the training constraint;
- `CHANNEL`: the re-entrant channel latitudes of the analytical `DinoConfiguration`;
- `batch`: the 16 normalized training states of the a posteriori data run (`optimal/data/<timestamp>/states.npy`, described by `DATA_ARGS`), `gens`: the generation of every run of the sweep `PREFIX`, `sources = {"data": batch} | gens`;
- `REFERENCE_DENSITY`: the batch-mean zonal sigma_0 section of the 200-state data batch, written once by `build_reference_density.py` (the only script with a DataLoader), the reference of the fig1 median member;
- `observer`: unnormalizes a batch; `to_dataset(x)` puts it on the T grid of `GRID` as an xarray Dataset.

The a posteriori runs are opened on the `NemoExperiment` of the 1/4° reference run by `aposteriori_experiment(results, files)`, their restarts by `restart_state(exp, path)`.

In [ ]:
%matplotlib inline
from common import *

In [ ]:
print(GRID.name, dict(GRID.namelist['nameos']))
print({s: tuple(x.shape) for s, x in sources.items()})
to_dataset(batch[:2])

## Shared computations

The sigma_0 sections and density error used by several figures, from `common.py`: TEOS-10 density (`metrics.get_density_at_surface`), volume-weighted zonal mean, and the vertical difference of sigma_0 with the xgcm grid of `GRID`.

In [ ]:
def sigma0(ds, exp=None):
    """TEOS-10 sigma_0 (float64) of the toce, soce of `ds`, NaN on the land of the tmask of `exp` (GRID)."""
    # In float32 the + 1000 of rho rounds it to 1.2e-4 kg m-3, which turns the near-zero vertical differences of the
    # data (pairs of levels with almost equal T, S) into ties and hides them from `density_error`.
    return teos_sigma0(ds.toce.astype(np.float64), ds.soce.astype(np.float64)).where((exp or GRID).mask.tmask == 1)


def zonal_mean(var, volume):
    """Volume-weighted mean along x_c of `var` over its ocean cells, NaN on rows without any."""
    return (var * volume).sum('x_c') / volume.where(var.notnull()).sum('x_c')


def zonal_sections(x):
    """Volume-weighted zonal mean of sigma_0 of a normalized batch x, (member, z_c - 1, y_c), last (land) level dropped."""
    section = zonal_mean(sigma0(to_dataset(x)), cell_volume(GRID))
    return section.transpose('member', 'z_c', 'y_c').values[:, :-1]


def density_error(x):
    """Percent of the ocean volume with a density inversion, per member of a normalized batch x.

    The vertical difference of sigma_0 is taken with the xgcm grid of GRID, as `Diagnostics.get_N_squared` does, onto
    the W points, and its negative values are weighted by e3w_0 e1t e2t. get_N_squared itself is not used: it clips
    N^2 below 1e-8 to 1e-7, which hides the inversions.
    """
    drho = GRID.grid.diff(sigma0(to_dataset(x)), 'Z', boundary='extend')
    mask = GRID.mask
    volume = (mask.e1t * mask.e2t * mask.e3w_0).where(drho.notnull(), 0.)
    return (100 * ((drho < 0) * volume).sum(('z_f', 'y_c', 'x_c')) / volume.sum(('z_f', 'y_c', 'x_c'))).values

## fig0_unsconstrained

Figure 0: one data state against one unconstrained state, as panel A of the "No constraint" tab of `machine/density_viewer.html`, drawn in the style of the paper maps.

Four figures in the `fig0_unconstrained` subfolder of FIGURES, each a pair of maps, data then unconstrained: T at the level nearest to 5 m, S at the level nearest to 136 m, and the level k-means clusters (`common.LevelClusters`, fit on the first N_FIT data states) of the same states at each of these levels, offset removed: the basin mean of the level, area-weighted (`xgcm.Grid.average` of GRID over X and Y).

In [ ]:
from fig0_unsconstrained import *


def anomalies(ds):
    """{field: (member, z, y, x)} of FIELDS of `ds` minus their area-weighted basin mean on each level, NaN on land, and
    {field: (member, z)} of these means."""
    means = {f: GRID.grid.average(ds[f], ['X', 'Y']) for f in FIELDS}
    return {f: (ds[f] - means[f]).values for f in FIELDS}, {f: means[f].values for f in FIELDS}

In [ ]:
tmask = GRID.mask.tmask.values.astype(bool)
depth = GRID.mask.gdept_1d.values
levels = [int(np.abs(depth - d).argmin()) for _, d in PANELS]
states = {s: to_dataset(sources[s][MEMBERS[s]:MEMBERS[s] + 1]) for s in SOURCES}
fit, fit_means = anomalies(to_dataset(batch[:N_FIT]))
clusters = LevelClusters(fit, tmask, levels, {f: m.mean(axis=0) for f, m in fit_means.items()})
labels = {s: clusters.labels({f: a[:, levels] for f, a in anomalies(states[s])[0].items()})[0] for s in SOURCES}
computed = {'levels': levels, 'depth': depth, 'labels': labels, 'k': clusters.k,
            'maps': {s: {f: states[s][f].values[0] for f in FIELDS} for s in SOURCES}}

In [ ]:
plot(computed);

## fig0_appendix

Figure 0, appendix: each layer of the unconstrained states has its own shift, as panel B of the "No constraint" tab of `machine/density_viewer.html`, for FIELD.

Left: basin mean of FIELD (area-weighted mean of each level, `xgcm.Grid.average` of GRID over X and Y) at the levels nearest to SHIFT_DEPTHS, one dot per state, N_STATES random data states and the unconstrained ones. Middle: correlation over the unconstrained members of the basin means of every pair of levels. Right: its mean against the number of levels apart, with the +-1/sqrt(n - 1) noise level of n members in grey.

In [ ]:
from fig0_appendix import *

In [ ]:
depth = GRID.mask.gdept_1d.values[:-1]
shift_levels = sorted({int(np.abs(depth - d).argmin()) for d in SHIFT_DEPTHS})
rng = np.random.default_rng(0)
states = {'data': batch[rng.choice(len(batch), N_STATES, replace=False)], SOURCE: sources[SOURCE]}
means = {s: GRID.grid.average(to_dataset(x)[FIELD], ['X', 'Y']).values[:, :-1] for s, x in states.items()}
C = np.corrcoef(means[SOURCE].T)
lags = np.arange(1, MAX_LAG + 1)
lag_mean = [np.nanmean(np.diagonal(C, offset=l)) for l in lags]
computed = {'depth': depth, 'shift_levels': shift_levels, 'means': means, 'C': C, 'lags': lags, 'lag_mean': lag_mean,
            'n': means[SOURCE].shape[0], 'rng': rng}

In [ ]:
plot(computed);

## fig1_density_profile

Figure 1: zonal sigma_0 sections (TEOS-10, `common.sigma0`), with the batch-average profile over the re-entrant channel latitudes (CHANNEL) as a black inset.

Main figure, split in two: batch-average section of each source in PAIRS, the two sources of a pair side by side (fig1a, fig1b), saved in the `fig1` subfolder of FIGURES. Appendix: every source in CONFIGS['fig1_appendix'], one per row, median member (upper median of the RMSE of its section to REFERENCE_DENSITY, the batch-mean section of the 200-state data batch, the criterion of the a posteriori selection) on the left and batch-average section on the right.

In [ ]:
from fig1_density_profile import *


def median_member(errors):
    """Index of the member at the median of `errors`, the upper median for an even batch."""
    return int(np.argsort(errors)[len(errors) // 2])

In [ ]:
depth = GRID.mask.gdept_1d.values[:-1]
lat = GRID.mask.gphit.mean('x_c').values
sections = {s: zonal_sections(sources[s]) for s in dict.fromkeys(CONFIGS['fig1_main'] + APPENDIX)}
medians = {}
for source, s in sections.items():
    errors = rmse(s, REFERENCE_DENSITY)
    medians[source] = median_member(errors)
    print(f'{source} : median member {medians[source]} / {len(errors)}, RMSE {errors[medians[source]]:.3f} '
          f'(mean {errors.mean():.3f})')
computed = {'depth': depth, 'lat': lat, 'sections': sections, 'medians': medians}

In [ ]:
plot(computed);

## fig2_state

Figure 2: the ocean state of each source as the A-priori tab of `machine/density_viewer.html`, maps drawn in the style of the paper maps.

One row per source, batch average. Columns:
- SST : mean over the batch of the surface temperature.
- Clusters : most frequent k-means cluster of the (T, S) profile of each column over the batch (`common.ProfileClusters`, fit on the first N_FIT data states, every state assigned to the nearest centre).
- Clusters in T-S : centres of the k-means (dashed) and mean profile of the columns of the batch in each cluster (solid, members weighted by their column count), one marker per level, sigma_0 isopycnals behind.
- T-S diagram : volume fraction (e1t e2t e3t_0 of GRID) of each (T, S) bin averaged over the batch, re-entrant channel (ocean cells between the CHANNEL latitudes of `DinoConfiguration`) in solar_r and the rest of the basin in greys, log10 scale, each normalised by its own volume, sigma_0 isopycnals.

Both T-S panels have S on the x axis and T on the y axis; the isopycnals are TEOS-10 sigma_0 (`teos_sigma0`).

Main figure: sources of CONFIGS['fig2_main']. Appendix: sources of CONFIGS['fig2_appendix'].

In [ ]:
from fig2_state import *


def fields(x):
    """Unnormalized toce, soce numpy arrays (member, z, y, x) of a normalized batch x, NaN on land."""
    ds = to_dataset(x)
    return {f: ds[f].values for f in FIELDS}


def batch_state(x, clusters):
    """
    Batch-average state of a normalized batch x:
        sst : mean SST (y, x), NaN on land
        cluster : most frequent cluster 1..N_CLUSTERS of `clusters` (y, x), NaN on land
        T, S : mean profile of the columns of each cluster (N_CLUSTERS, z - 1), members weighted by their column count
        ts : mean volume fraction in each (T, S) bin of the channel and of the rest of the basin (2, nT, nS)
    """
    tmask = GRID.mask.tmask.values.astype(bool)
    channel = tmask & ((GRID.mask.gphit >= CHANNEL[0]) & (GRID.mask.gphit <= CHANNEL[1])).values[None]
    volume = cell_volume(GRID).values
    sst, counts, ts, profile_sum, profile_weight = 0., 0., 0., 0., 0.
    for i in range(0, len(x), CHUNK):
        f = fields(x[i:i + CHUNK])
        sst = sst + f['toce'][:, 0].sum(axis=0)
        labels = clusters.labels(f)
        counts = counts + np.stack([(labels == k + 1).sum(axis=0) for k in range(N_CLUSTERS)])
        means, columns = clusters.mean_profiles(f, labels)
        w = np.where(np.isfinite(means), columns[:, :, None, None], 0)
        profile_sum = profile_sum + (np.nan_to_num(means) * w).sum(axis=0)
        profile_weight = profile_weight + w.sum(axis=0)
        ts = ts + np.stack([ts_histograms(f, channel, volume), ts_histograms(f, tmask & ~channel, volume)],
                           axis=1).sum(axis=0)
    profiles = np.where(profile_weight > 0, profile_sum / np.maximum(profile_weight, 1), np.nan)
    return {'sst': np.where(tmask[0], sst / len(x), np.nan),
            'cluster': np.where(tmask[0], counts.argmax(axis=0) + 1., np.nan),
            'T': profiles[:, 0], 'S': profiles[:, 1], 'ts': ts / len(x)}

In [ ]:
clusters = ProfileClusters(fields(batch[:N_FIT]), GRID.mask.tmask.values.astype(bool))
shown = [s for s in dict.fromkeys(CONFIGS['fig2_main'] + CONFIGS['fig2_appendix']) if s in sources]
states = {s: batch_state(sources[s], clusters) for s in shown}
computed = {'centres': clusters.centres, 'states': states}

In [ ]:
plot(computed);

## fig4_constraint_strenght

Figure 4: strength of the increasing density constraint (negative_grad_density_huber, C7) as a function of eta.

Top: on twin y axes, density error (`common.density_error`, percent of the ocean volume with a sigma_0 inversion, mean +/- std over the batch) and RMSE of the batch mean to the data (RMS over ocean cells of the difference between the batch-average and the data-average sigma_0), of the b8 eta sweep CONFIGS["fig4_sweep"]. eta 0 is the paired unconstrained run (boundary projection only, same seed and batch). The data references are dotted: its density error, and for the RMSE, computed on b8 batches against all the data states (`batch`), the mean RMSE of N_SUBSETS random 8-member subsets of the data. Bottom: batch-average zonal sigma_0 section, with its profile over the channel latitudes as inset (as fig1), of the unconstrained run and of the ETAS_FRIEZE runs.

In [ ]:
from fig4_constraint_strenght import *


def load_sweep(sweep=SWEEP):
    """Generation of every run of `sweep`, keyed by its eta; for a repeated eta the latest timestamp is kept."""
    runs = {}
    for path in by_timestamp(glob(f'{sweep}/*/generated.npy')):
        with open(f'{os.path.dirname(path)}/args.json') as f:
            runs[json.load(f)['eta']] = torch.from_numpy(np.load(path))
    return dict(sorted(runs.items()))


def ocean_sigma0(x):
    """sigma_0 (float32) of a normalized batch x on the ocean cells of GRID, (member, n_ocean)."""
    return sigma0(to_dataset(x)).values[:, GRID.mask.tmask.values == 1].astype(np.float32)

In [ ]:
sweep = load_sweep()
none_b8 = torch.from_numpy(np.load(f'{NONE_B8}/generated.npy'))
runs = {0: none_b8} | sweep
sigma_data = ocean_sigma0(batch)
rmse_to_data = lambda s: np.sqrt(np.mean((s.mean(0) - sigma_data.mean(0)) ** 2))
rng = np.random.default_rng(0)
rmse_data = np.mean([rmse_to_data(sigma_data[rng.choice(len(sigma_data), len(none_b8), replace=False)])
                     for _ in range(N_SUBSETS)])
errors = {eta: density_error(x) for eta, x in runs.items()}
rmses = {eta: rmse_to_data(ocean_sigma0(x)) for eta, x in runs.items()}
error_data = density_error(batch).mean()
for eta in runs:
    print(f'eta {eta:g} : density error {errors[eta].mean():.2f} +/- {errors[eta].std(ddof=1):.2f}, '
          f'RMSE {rmses[eta]:.3f}')
print(f'data : density error {error_data:.3f}, RMSE of b{len(none_b8)} subsets {rmse_data:.3f}')
frieze = [(none_b8, 'No constraint')] + [(sweep[eta], rf'$\eta = {eta:g}$') for eta in ETAS_FRIEZE]
computed = {'errors': errors, 'rmses': rmses, 'error_data': error_data, 'rmse_data': rmse_data,
            'frieze': [(np.nanmean(zonal_sections(x), axis=0), title) for x, title in frieze],
            'depth': GRID.mask.gdept_1d.values[:-1], 'lat': GRID.mask.gphit.mean('x_c').values}

In [ ]:
plot(computed);

## table1_score_table

Table 1: mean over the batch of the box metrics and of the density error, one row per source, ordered and named as in NAMES.

Box metrics: volume mean (e1t e2t e3t_0 of GRID) of T and S over the Bottom Water box (below DEPTH_BOX or poleward of 30°) and the Deep Water box (between 500 m and DEPTH_BOX, within 30° of the equator), the depth of a cell being its gdept_0. Density error: `common.density_error`. The LaTeX table rounds mean and std to one decimal and gives the box metrics of the generated sources as their bias to the data mean.

In [ ]:
from table1_score_table import *


def scores(x):
    """Per-member value (member,) of every column of COLUMNS for a normalized batch x."""
    ds = to_dataset(x)
    lat, depth = np.abs(GRID.mask.gphit), GRID.mask.gdept_0
    boxes = {'BWbox': ~((depth < DEPTH_BOX) & (lat < 30)), 'DWbox': (depth > 500) & (depth < DEPTH_BOX) & (lat < 30)}
    volume = cell_volume(GRID)
    out = {(b, f): ds[f].weighted(volume.where(box, 0.)).mean(('z_c', 'y_c', 'x_c')).values
           for b, box in boxes.items() for f in ('toce', 'soce')}
    return out | {('density_errors', 'density'): density_error(x)}

In [ ]:
order = [s for s in CONFIGS['table'] if s in sources]
values = {s: scores(sources[s]) for s in order}
computed = {'order': order, 'means': {s: {c: v.mean() for c, v in values[s].items()} for s in order},
            'stds': {s: {c: v.std(ddof=1) for c, v in values[s].items()} for s in order}}

In [ ]:
table(computed);

## table2_moments

Table 2: first two moments of each generation against the whole training set, per field (T, S, SSH).

For a source of n states, at every ocean cell, the batch mean mu and the batch variance var (ddof = 1) are compared to those of all the training states (mu_D, var_D):

    bias   = sqrt(sum w (mu - mu_D)^2 / sum w)          [field units]
    spread = sqrt(sum w var / sum w var_D)              [1 = same variance, < 1 under-dispersive]

with w the ocean cell volume e1t e2t e3t_0 of GRID (area e1t e2t of the surface ocean cells for SSH). The noise floor is the same scores for N_FLOOR disjoint random groups of n training states (mean ± std over the groups), the data row the first n states of `batch`, the data states of the a posteriori data run, drawn from the start of the time-ordered training set.

In [ ]:
from table2_moments import *


def scores(mean, var, reference, w):
    """(bias, spread) of a (mean, var) pair against the reference (mean, var), weighted by w."""
    bias = np.sqrt((w * (mean - reference[0]) ** 2).sum() / w.sum())
    spread = np.sqrt((w * var).sum() / (w * reference[1]).sum())
    return np.array([bias, spread])


def batch_moments(x):
    """Mean and variance (ddof = 1) over the batch of every field of a normalized batch x, land as 0."""
    ds = to_dataset(x).fillna(0.).astype(np.float64)
    return {f: (ds[f].mean('member').values, ds[f].var('member', ddof=1).values) for f in FIELDS}

In [ ]:
n = next(iter(gens.values())).shape[0]
volume = cell_volume(GRID).values
w = {'toce': volume, 'soce': volume, 'ssh': (GRID.mask.e1t * GRID.mask.e2t * GRID.mask.tmask.isel(z_c=0)).values}
full, floor = training_moments(n, N_FLOOR)

rows = {'floor': {}, 'data_seed': {}}
for f in FIELDS:
    group_scores = np.array([scores(floor[f][0][g], floor[f][1][g], full[f], w[f]) for g in range(N_FLOOR)])
    rows['floor'][f] = (group_scores.mean(0), group_scores.std(0))
for name, x in [('data_seed', batch[:n])] + list(gens.items()):
    m = batch_moments(x)
    rows[name] = {f: (scores(*m[f], full[f], w[f]), None) for f in FIELDS}
computed = {'n': n, 'rows': rows}

In [ ]:
table(computed);

## fig5_aposteriori

Figure 5: a posteriori behaviour of the generated states in the 1/4° NEMO model, recomputed from the NEMO outputs of the runs of `notebooks/a_posteriori/` (experiment EXPERIMENT, run started from the median member of each generation), on the dinostics `NemoExperiment` of the 1/4° reference grid.

Left, one row per method of METHODS:
- zonal mean sigma_0 (TEOS-10, `common.sigma0`, volume-weighted with the time-varying e3t of the outputs) after 2 months (last daily output), with its channel profile as inset (as fig1),
- its difference to the initial state (day 0, the restart the run starts from, e3t_0),
- its difference to the reference density, the mean zonal sigma_0 section of the data states (`batch`, 1° grid, the 1/4° section interpolated onto its latitudes, both grids share their levels).

Right, over time: a) transport through the re-entrant channel, `Diagnostics.get_ACC` (uoce e3u_0 e2u through the first U column), b) RMSE of the zonal sigma_0 section to the reference density (grey band: RMSE of each data state to it, min-max), c) empty. Also saved in the `aposteriori` subfolder of FIGURES: density_2m, the zonal sigma_0 after 2 months of each method of ROW_METHODS side by side, as fig1. Dashed: the 1/4° spin-up restart. A method whose run is not collected is skipped.

In [ ]:
from fig5_aposteriori import *


def restart(path):
    """Zonal sigma_0 section (z_c, y_c) and channel transport [Sv] of the 1/4° restart `path`."""
    exp = aposteriori_experiment()
    state = restart_state(exp, path)
    exp.data['U_3D'] = state
    section = zonal_mean(sigma0(state, exp), cell_volume(exp))
    return section.transpose('z_c', 'y_c').values, float(exp.diagnostics.get_ACC().values[0])


def run(folder):
    """Days from the start, zonal sigma_0 sections (frames, z_c, y_c) and channel transports [Sv] of the EXPERIMENT run
    in `folder`. Frame 0 is the restart the run starts from (instantaneous), frame i >= 1 the i-th daily mean."""
    paths = experiment_paths(EXPERIMENT)
    exp = aposteriori_experiment(f'{folder}/{paths["results"]}', files=('T_3D', 'U_3D'))
    T = exp.data['T_3D']
    days = [(t - T.t_bounds.values[0, 0]).total_seconds() / 86400 for t in T.t.values]
    sections = zonal_mean(sigma0(T, exp), cell_volume(exp, T.e3t)).transpose('t', 'z_c', 'y_c').values
    section0, transport0 = restart(f'{folder}/{paths["restart_1_4deg"]}')
    return (np.r_[0., days], np.concatenate([section0[None], sections]),
            np.r_[transport0, exp.diagnostics.get_ACC().values])


def to_lat(section, lat, lat_to):
    """Section (z, y) on `lat` linearly interpolated onto `lat_to`, NaN where a neighbour is NaN."""
    def interp(a):
        finite = np.isfinite(a)
        v = np.interp(lat_to, lat, np.where(finite, a, 0.))
        w = np.interp(lat_to, lat, finite.astype(float))
        return np.where(w > 1 - 1e-6, v, np.nan)
    return np.stack([interp(row) for row in section])

In [ ]:
lat_1 = GRID.mask.gphit.mean('x_c').values
data_sections = zonal_sections(batch)
reference = np.nanmean(data_sections, axis=0)
data_rmse = rmse(data_sections, reference)

grid = aposteriori_experiment()
lat, depth = grid.mask.gphit.mean('x_c').values, grid.mask.gdept_1d.values
nz = reference.shape[0]
rmse_to_reference = lambda section: float(rmse(to_lat(section[:nz], lat, lat_1)[None], reference)[0])
spinup_path = sorted(glob(f'{REFERENCES}{GRID_1_4DEG}/{EXP}_*_restart.nc'))[-1]
spinup_section, spinup_transport = restart(spinup_path)
spinup_rmse = rmse_to_reference(spinup_section)
print(f'spin-up {spinup_path} : transport {spinup_transport:.1f} Sv, RMSE {spinup_rmse:.3f}')

folders = collected_runs(METHODS, EXPERIMENT)
runs = {}
for method, folder in folders.items():
    days, sections, transports = run(folder)
    runs[method] = dict(days=days, sections=sections, transport=transports,
                        rmse=np.array([rmse_to_reference(s) for s in sections]))
    print(f'{method} : {folder}, day {days[-1]:g}, RMSE to reference {runs[method]["rmse"][-1]:.3f}, '
          f'transport {transports[-1]:.1f} Sv')
computed = {'folders': folders, 'spinup_path': spinup_path, 'runs': runs,
            'vs_day0': {m: r['sections'][-1] - r['sections'][0] for m, r in runs.items()},
            'vs_reference': {m: to_lat(r['sections'][-1][:nz], lat, lat_1) - reference for m, r in runs.items()},
            'spinup_transport': spinup_transport, 'spinup_rmse': spinup_rmse, 'data_rmse': data_rmse,
            'lat': lat, 'depth': depth, 'lat_1': lat_1, 'depth_1': depth[:nz]}

In [ ]:
plot(computed);

## fig6_amoc

Figure 6: depth-space meridional overturning stream function of the a posteriori runs in the 1/4° NEMO model, one panel per configuration of `NAMES` whose EXPERIMENT run is collected, averaged over the last YEARS years of the run.

    psi(z, y) = sum_{k' <= k} sum_x <v e3v> e1v  [Sv]

<v e3v> is `voce_e3v`, the time mean of the thickness-weighted meridional velocity written by XIOS, multiplied by e1v as `Diagnostics.get_MOC` does before its transform to density classes; here the sum over x is cumulated over the levels from the surface down instead, on the dinostics `NemoExperiment` of the 1/4° reference grid. psi is NaN on the levels with no wet V point along the zonal line. The y axis is the nominal level depth gdept_1d: DINO is in s-coordinates, so below ~1000 m the actual depth of a level is shallower where the bottom is shallower. Also saved in the `aposteriori` subfolder of FIGURES: amoc, the methods of ROW_METHODS side by side. The averaging, colour scale and panel layout are shared with `fig6_moc_density.py`.

In [ ]:
from fig6_amoc import *


def mean_moc(results, years=YEARS):
    """Depth-space stream function [Sv] (z_c, y_f) of the V outputs of `results` averaged over the last `years` years,
    and the length of the average in years."""
    exp = aposteriori_experiment(results, files=('V_3D',))
    V = exp.data['V_3D']
    last, length = last_outputs(V, years)
    transport = (V.voce_e3v * exp.domain.e1v).isel(t=slice(-last, None)).mean('t')
    psi = transport.sum('x_c').cumsum('z_c') / 1e6
    return psi.where(exp.mask.vmask.any('x_c')).transpose('z_c', 'y_f').compute(), length

In [ ]:
grid = aposteriori_experiment()
lat, depth = grid.domain.gphiv.mean('x_c').values, grid.mask.gdept_1d.values
folders = collected_runs(NAMES, EXPERIMENT)
mocs, lengths = {}, {}
for method, folder in folders.items():
    results = f'{folder}/{experiment_paths(EXPERIMENT)["results"]}'
    mocs[method], lengths[method] = mean_moc(results)
    print(f'{method} : {results}, mean of {lengths[method]:g} years, '
          f'psi in [{float(mocs[method].min()):.1f}, {float(mocs[method].max()):.1f}] Sv')
computed = {'folders': folders, 'mocs': mocs, 'lengths': lengths, 'lat': lat, 'depth': depth}

In [ ]:
plot(computed);

## fig6_moc_density

Figure 6, density-space counterpart: meridional overturning stream function of the a posteriori runs in the 1/4° NEMO model in sigma_2 classes, shown at its pseudo-depth, one panel per configuration of `NAMES` whose EXPERIMENT run is collected, averaged over the last YEARS years of the run. Computed with `Diagnostics.get_MOC_pseudodepth`.

    psi(rho, y) = sum_{rho' <= rho} sum_x T(rho', y, x)  [Sv]

T(rho', y, x) is the transport <v e3v> e1v of `voce_e3v`, remapped conservatively by xgcm onto the density classes of `GridManipulation.transform_to_density`, rho = S-EOS density referenced at z = Z_REF (rho_ref = RHO_REF) of the daily toce, soce interpolated on the V points. Its class edges are fixed, from rho_ref - 4 to rho_ref + 1.67 kg m-3: water outside them is dropped from the transform. get_MOC_pseudodepth maps each class to the depth z* where the volume of the zonal line below z* (from the bathymetry of e3v_0) equals the volume denser than the class:

    sum_x area (H - z*)^+ = sum_x sum_{rho' >= rho} V(rho', y, x)

The MOC and its pseudo-depth are computed for every daily output of the last YEARS years and averaged over time in density space. The y axis of the depth figures is the pseudo-depth, stretched as fig6_amoc.py; the sigma figures show psi against the class index, labelled in sigma_2 = rho - 1000. The averaging (YEARS of the last outputs), colour scale and panel layout are those of `fig6_amoc.py`.

In [ ]:
from fig6_moc_density import *


def mean_moc_density(results, years=YEARS):
    """Density-space stream function [Sv] (y_f, rho) of the daily outputs of `results` and pseudo-depth [m] (y_f, rho)
    of its classes, both averaged over the last `years` years, and the length of the average in years."""
    exp = experiment(results)
    V = exp.data['V_3D']
    last, length = last_outputs(V, years)
    # 'vocetr_eff' is the name get_MOC_pseudodepth takes as a transport already, voce_e3v holding e3v.
    transport = (V.voce_e3v * exp.domain.e1v).rename('vocetr_eff')
    e3v = exp.domain.e3v_0.expand_dims(t=V.t)
    psi, depth = 0, 0
    # One day per call: over several days the (t, y, rho, z) mask of get_MOC_pseudodepth takes tens of GB.
    for t in range(V.sizes['t'] - last, V.sizes['t']):
        moc, z = exp.diagnostics.get_MOC_pseudodepth(transport, e3v, isel={'t': t}, z=Z_REF)
        psi, depth = psi + moc.compute() / last, depth + z.compute() / last
    order = lambda a: a.drop_vars('t', errors='ignore').transpose('y_f', 'rho')
    return order(psi), order(depth), length

In [ ]:
lat = aposteriori_experiment().domain.gphiv.mean('x_c').values
folders = collected_runs(NAMES, EXPERIMENT)
mocs, depths, lengths = {}, {}, {}
for method, folder in folders.items():
    results = f'{folder}/{experiment_paths(EXPERIMENT)["results"]}'
    mocs[method], depths[method], lengths[method] = mean_moc_density(results)
    print(f'{method} : {results}, mean of {lengths[method]:g} years, psi in [{float(mocs[method].min()):.1f}, '
          f'{float(mocs[method].max()):.1f}] Sv, pseudo-depth in [{float(depths[method].min()):.0f}, '
          f'{float(depths[method].max()):.0f}] m')
sigma = next(iter(mocs.values()))['rho'].values - 1000
computed = {'folders': folders, 'mocs': mocs, 'depths': depths, 'lengths': lengths, 'sigma': sigma, 'lat': lat}

In [ ]:
plot(computed);

## fig7_aposteriori_drift

Figure 7: drift of the temperature and salinity on each level in the a posteriori runs of fig5 (experiment EXPERIMENT, 1/4° NEMO run started from the median member of each generation of the sweep PREFIX).

For each method of METHODS (those of fig5, with its COLORS), the RMSE on each level of T and S to the initial state (day 0, the restart the run starts from), weighted by the cell volume e1t e2t e3t of each output (time-varying e3t), averaged over the last LAST_DAYS daily outputs of the run, then smoothed along depth with a Gaussian of SMOOTHING levels. One line per method, one panel per field, saved as two files in the `fig7` subfolder of FIGURES. A method whose run is not collected is skipped.

In [ ]:
from fig7_aposteriori_drift import *


def drift(folder, last_days):
    """Drift (z_c,) of each field of FIELDS of the EXPERIMENT run in `folder`: per level, the volume-weighted RMSE of
    each of the last `last_days` daily outputs to the restart it starts from, averaged over them, smoothed along depth
    with a Gaussian of SMOOTHING levels over the ocean levels, NaN on land levels."""
    paths = experiment_paths(EXPERIMENT)
    exp = aposteriori_experiment(f'{folder}/{paths["results"]}', files=('T_3D',))
    T = exp.data['T_3D'].isel(t=slice(-last_days, None))
    initial = restart_state(exp, f'{folder}/{paths["restart_1_4deg"]}')
    volume = cell_volume(exp, T.e3t)
    out = {}
    for f in FIELDS:
        d = np.sqrt(((T[f] - initial[f]) ** 2).weighted(volume).mean(('y_c', 'x_c'))).mean('t').values
        ocean = np.isfinite(d)
        out[f] = np.full_like(d, np.nan)
        out[f][ocean] = gaussian_filter1d(d[ocean], SMOOTHING, mode='nearest')
    return out

In [ ]:
methods = METHODS
folders = collected_runs(methods, EXPERIMENT)
drifts = {}
for method, folder in folders.items():
    print(f'{method} : {folder}')
    drifts[method] = drift(folder, LAST_DAYS)
    for f in FIELDS:
        print(f'  {f} : max RMSE {np.nanmax(drifts[method][f]):.4f} at level {int(np.nanargmax(drifts[method][f]))}')
computed = {'folders': folders, 'drifts': drifts, 'depth': aposteriori_experiment().mask.gdept_1d.values}

In [ ]:
plot(computed);

## fig7_appendix

Figure 7, appendix: the drift of figure 7 with, in addition, the unconstrained run with BorderZero projection (`none_border0`), labelled as in `common.NAMES`. Same computation and layout as `fig7_aposteriori_drift.py`; saved as `fig7_appendix_drift_{field}.pdf` in the `fig7` subfolder of FIGURES.

In [ ]:
from fig7_appendix import *

In [ ]:
computed = fig7.compute(METHODS)

In [ ]:
plot(computed);

## fig8_drift_spatial

Figure 8: map of the temperature drift over the first days of the a posteriori runs of fig5 and fig7 (experiment EXPERIMENT, 1/4° NEMO run started from the median member of each generation of the sweep PREFIX).

For each method of METHODS, side by side: at every water column, the thickness-weighted RMSE over depth of T to the initial state (day 0, the restart the run starts from), with the time-varying e3t of each output,

    sqrt(sum_z e3t (T(x, y, z) - T_init(x, y, z))^2 / sum_z e3t),

averaged over the first FIRST_DAYS daily outputs, divided by its area-weighted mean over the domain (`xgcm.Grid.average` of the 1/4° experiment over X and Y, given in the title of each panel). One logarithmic colour scale for all the panels, centred on 1 (the domain mean). A method whose run is not collected is skipped.

In [ ]:
from fig8_drift_spatial import *


def column_rmse(folder, experiment, n_days):
    """Thickness-weighted RMSE over depth (y_c, x_c) of T to the restart the `experiment` run in `folder` starts from,
    averaged over its first `n_days` daily outputs, NaN on land columns, and its area-weighted domain mean."""
    paths = experiment_paths(experiment)
    exp = aposteriori_experiment(f'{folder}/{paths["results"]}', files=('T_3D',))
    T = exp.data['T_3D'].isel(t=slice(0, n_days))
    initial = restart_state(exp, f'{folder}/{paths["restart_1_4deg"]}')
    e3t = T.e3t.where(exp.mask.tmask == 1.0, 0.)
    rmse_map = np.sqrt(((T.toce - initial.toce) ** 2).weighted(e3t).mean('z_c')).mean('t').compute()
    return rmse_map.values, float(exp.grid.average(rmse_map, ['X', 'Y']))

In [ ]:
methods = METHODS
folders = collected_runs(methods, EXPERIMENT)
maps, means = {}, {}
for method, folder in folders.items():
    maps[method], means[method] = column_rmse(folder, EXPERIMENT, FIRST_DAYS)
    print(f'{method} : {folder}, domain mean {means[method]:.3f} °C')
computed = {'folders': folders, 'maps': maps, 'means': means}

In [ ]:
plot(computed);

## fig8_appendix

Figure 8, appendix: the drift maps of figure 8 with, in addition, the unconstrained run with BorderZero projection (`none_border0`), titled as in `common.NAMES`, as an extra panel on the right. Same computation and layout as `fig8_drift_spatial.py`; saved as `fig8_appendix_drift_spatial.pdf` in FIGURES.

In [ ]:
from fig8_appendix import *

In [ ]:
computed = fig8.compute(METHODS)

In [ ]:
plot(computed);

## fig8_sweep_density_huber

Figure 8, density sweep: the temperature drift over the first days of the a posteriori runs started from the sweep of the increasing-density constraint strength eta, and the density error of their initial states (experiment EXPERIMENT, 1/4° NEMO run started from the median member of each generation, minimal restart).

Runs: the sweep SWEEP_PREFIX, with eta read from `selection.json`, and the unconstrained run with BorderZero projection (`none_border0` of the sweep ZERO_PREFIX) as eta = 0. A run that is not collected is skipped.

`fig8_sweep_density_huber.pdf`: maps as `fig8_drift_spatial.py`, one panel per run by increasing eta, on rows of PANELS_PER_ROW.

`fig8_sweep_density_huber_vs_eta.pdf`: against eta (symlog axis), the domain mean of the maps and, on a second y axis, the density error E (`common.density_error`) of the selected median member (top), and the error of the initial section (bottom): the `rmse` of `selection.json`, RMSE of the zonal sigma_0 section of the selected median member to the batch-mean section of a data batch, and `rmse_mean`, its mean over the member batch (both computed by the a posteriori selection step).

`fig8_correlation.pdf`: domain mean of the maps against the density error E of the selected median member (log axis), one point per run up to eta = CORRELATION_ETA_MAX, labelled with its eta.

In [ ]:
from fig8_sweep_density_huber import *


def load_runs():
    """Runs collected for EXPERIMENT, by increasing eta: a list of dicts with the aposteriori `folder`, `eta`, the
    FIRST_DAYS mean column RMSE `map` (j, i), its domain `mean`, the `rmse` and `rmse_mean` of the selection, and the
    density error `E` of the selected member."""
    folders = (collected_runs(None, EXPERIMENT, prefix=SWEEP_PREFIX)
               | collected_runs([ZERO_CONSTRAINT], EXPERIMENT, prefix=ZERO_PREFIX))
    runs = []
    for constraint, folder in folders.items():
        selection = read_json(f'{folder}/{SELECTION}')
        rmse_map, mean = column_rmse(folder, EXPERIMENT, FIRST_DAYS)
        generated = np.load(f'{os.environ["OCEANDATA"]}/{selection["generation_rel"]}/generated.npy', mmap_mode='r')
        member = torch.from_numpy(np.array(generated[selection['member']:selection['member'] + 1]))
        runs.append(dict(folder=folder, eta=selection.get('eta', 0.), map=rmse_map, mean=mean, rmse=selection['rmse'],
                         rmse_mean=selection['rmse_mean'], E=float(density_error(member)[0])))
        print(f'{constraint} : {folder}, domain mean {mean:.3f} °C')
    return sorted(runs, key=lambda run: run['eta'])

In [ ]:
runs = load_runs()
print('eta, domain mean drift (°C), selection rmse, rmse_mean, density error E (%)')
for run in runs:
    print(f'{run["eta"]:g}, {run["mean"]:.4f}, {run["rmse"]:.4f}, {run["rmse_mean"]:.4f}, {run["E"]:.3f}')
computed = {'runs': runs}

In [ ]:
plot(computed);